# Flickr30k Teacher handoff: ExpansionNet v2 → S1

Notebook này chỉ khóa handoff và phương pháp; không đọc ảnh, không huấn luyện và không mở Test. Ba notebook T1/T2/T3 độc lập, có thể chạy song song.

## Kaggle input bắt buộc — đúng 4 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_S_Audit`.
4. Output `phase3_teacher_a_expansionnet_v2_handoff`.

Không gắn dataset ảnh, `pycocoevalcap`, MS COCO, output KD hoặc caption Test.

## Runtime

- Accelerator: None/CPU.
- Internet: Off.
- Dự kiến: 1–5 phút; T2/T3 có thể lâu hơn do kiểm SHA-256 checkpoint.


## Cell 1 — Xác minh Data Contract, Student audit và Test seal

In [1]:
EXPERIMENT_ID = "F30K_T1_Handoff"

import csv
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working"))
OUTPUT_ROOT = WORK_ROOT / EXPERIMENT_ID
CONFIG_DIR = OUTPUT_ROOT / "configs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in (OUTPUT_ROOT, CONFIG_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def unique_file(name):
    matches = sorted(
        path for path in INPUT_ROOT.rglob(name)
        if path.is_file() and OUTPUT_ROOT not in path.parents
    )
    assert len(matches) == 1, (
        f"Cần đúng một file {name}. Hãy chỉ gắn các input được liệt kê.",
        [str(path) for path in matches],
    )
    return matches[0]

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

DATA_CONTRACT_PATH = unique_file("F30K_DATA_CONTRACT.json")
BASE_READY_PATH = unique_file("F30K_BASE_READY.json")
STUDENT_AUDIT_PATH = unique_file("F30K_STUDENT_BASELINE_AUDIT.json")

data_contract = read_json(DATA_CONTRACT_PATH)
base_ready = read_json(BASE_READY_PATH)
student_audit = read_json(STUDENT_AUDIT_PATH)

assert data_contract["status"] == "complete"
assert data_contract["dataset"] == "Flickr30k"
assert data_contract["train_image_count"] == 29000
assert data_contract["validation_image_count"] == 1014
assert data_contract["test_image_count"] == 1000
assert data_contract["vocabulary_size"] == 7456
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False

assert base_ready["status"] == "authorized"
assert base_ready["parallel_execution_authorized"] is True
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False

assert student_audit["status"] == "passed"
assert student_audit["verified_run_count"] == 9
assert student_audit["expected_run_count"] == 9
assert student_audit["test_set_status"] == "sealed"
assert student_audit["test_set_used"] is False
assert student_audit["next_phase_authorized"] == (
    "Flickr30k Teacher T1/T2/T3 evaluation or adaptation"
)

UPSTREAM = {
    "data_contract": {
        "path": str(DATA_CONTRACT_PATH),
        "sha256": sha256(DATA_CONTRACT_PATH),
    },
    "base_ready": {
        "path": str(BASE_READY_PATH),
        "sha256": sha256(BASE_READY_PATH),
    },
    "student_audit": {
        "path": str(STUDENT_AUDIT_PATH),
        "sha256": sha256(STUDENT_AUDIT_PATH),
    },
}

print("=" * 78)
print(f"{EXPERIMENT_ID} — COMMON FLICKR30K GATE")
print("=" * 78)
print("Data Contract             : COMPLETE")
print("Student baselines         : 9 / 9 VERIFIED")
print("Train / Validation        : 29,000 / 1,014 images")
print("Flickr30k vocabulary      : 7,456")
print("Held-out Test             : SEALED / NOT READ")
print("Parallel Teacher handoff  : AUTHORIZED")


F30K_T1_Handoff — COMMON FLICKR30K GATE
Data Contract             : COMPLETE
Student baselines         : 9 / 9 VERIFIED
Train / Validation        : 29,000 / 1,014 images
Flickr30k vocabulary      : 7,456
Held-out Test             : SEALED / NOT READ
Parallel Teacher handoff  : AUTHORIZED


## Cell 2 — Khóa nguồn Teacher và hợp đồng KD trên Flickr30k

In [2]:

EXPERIMENT_ID = "F30K_T1_Handoff"
TEACHER_ID = "T1"
TEACHER_MODEL = "ExpansionNet v2"
MATCHED_STUDENT = "S1 MobileNetV3-Large"
STATUS = "passed"

handoff_path = unique_file(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
registry_path = unique_file("teacher_a_expansionnet_v2_artifact_registry.json")
handoff = read_json(handoff_path)
registry = read_json(registry_path)

assert handoff["status"] == "complete"
assert handoff["teacher"] == "ExpansionNet v2"
assert handoff["matched_student"] == "S1 MobileNetV3-Large"
assert handoff["selected_method"] == "sequence_kd"
assert handoff["direct_logit_kd_allowed"] is False
assert handoff["test_set_status"] == "sealed"
assert handoff["test_set_used"] is False
assert len(registry["teacher_checkpoint_sha256"]) == 64

TEACHER_SOURCE = {
    "handoff_path": str(handoff_path),
    "handoff_sha256": sha256(handoff_path),
    "registry_path": str(registry_path),
    "registry_sha256": sha256(registry_path),
    "checkpoint_sha256_locked": registry["teacher_checkpoint_sha256"],
    "mscoco_pseudocaptions_reused_for_flickr30k": False,
}
METHOD_CONTRACT = {
    "selected_method": "Sequence KD",
    "direct_logit_kd_on_f30k": False,
    "reason": (
        "ExpansionNet v2 uses its native vocabulary. Only generated sequences "
        "may supervise the Flickr30k Student."
    ),
    "flickr30k_pseudocaptions_required": True,
    "flickr30k_train_images_expected": 29000,
    "decoding": {"beam_size": 3, "length_penalty": 0.7},
}
NEXT_PHASE = "F30K_T1_Pseudo"

UPSTREAM["teacher_handoff"] = {
    "path": str(handoff_path), "sha256": sha256(handoff_path)
}
UPSTREAM["teacher_registry"] = {
    "path": str(registry_path), "sha256": sha256(registry_path)
}


## Cell 3 — Xuất handoff, contract và manifest SHA-256

In [3]:

created_at = datetime.now(timezone.utc).isoformat()
result = {
    "schema_version": "f30k_teacher_handoff_v1",
    "created_at_utc": created_at,
    "experiment_id": EXPERIMENT_ID,
    "status": STATUS,
    "teacher_id": TEACHER_ID,
    "teacher_model": TEACHER_MODEL,
    "matched_student": MATCHED_STUDENT,
    "flickr30k": {
        "train_images": 29000,
        "validation_images": 1014,
        "test_images": 1000,
        "vocabulary_size": 7456,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
    "method_contract": METHOD_CONTRACT,
    "teacher_source": TEACHER_SOURCE,
    "upstream": UPSTREAM,
    "next_phase_authorized": NEXT_PHASE,
    "parallel_execution_group": "F30K_TEACHER_HANDOFF",
}

result_path = OUTPUT_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
contract_path = CONFIG_DIR / f"{EXPERIMENT_ID}_METHOD_CONTRACT.json"
result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
contract_path.write_text(
    json.dumps(METHOD_CONTRACT, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

artifacts = []
for path in (result_path, contract_path):
    artifacts.append({
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })

manifest = {
    "schema_version": "f30k_teacher_handoff_manifest_v1",
    "created_at_utc": created_at,
    "experiment_id": EXPERIMENT_ID,
    "status": STATUS,
    "artifacts": artifacts,
    "test_set_status": "sealed",
    "test_set_used": False,
}
manifest_json = MANIFEST_DIR / f"{EXPERIMENT_ID}_MANIFEST.json"
manifest_json.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
manifest_csv = MANIFEST_DIR / f"{EXPERIMENT_ID}_MANIFEST.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(artifacts)

print("=" * 78)
print(f"{EXPERIMENT_ID} — FINAL")
print("=" * 78)
print(f"Status                    : {STATUS}")
print(f"Teacher                   : {TEACHER_MODEL}")
print(f"Matched Student           : {MATCHED_STUDENT}")
print(f"Locked KD method          : {METHOD_CONTRACT['selected_method']}")
print(f"Direct Logit-KD on F30K   : {METHOD_CONTRACT['direct_logit_kd_on_f30k']}")
print(f"Next                      : {NEXT_PHASE}")
print("Held-out Test             : SEALED / NOT READ")
print(f"Output                    : {OUTPUT_ROOT}")


F30K_T1_Handoff — FINAL
Status                    : passed
Teacher                   : ExpansionNet v2
Matched Student           : S1 MobileNetV3-Large
Locked KD method          : Sequence KD
Direct Logit-KD on F30K   : False
Next                      : F30K_T1_Pseudo
Held-out Test             : SEALED / NOT READ
Output                    : /kaggle/working/F30K_T1_Handoff
